Entry: EMA(30) rising AND Close breaks above MAX(7) AND NATR(7) rising

Exit: Close < MIN(7)

In [7]:
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'ADAUSDT'],
    'INTERVALS': ['1h', '4h', '1d'],
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002,
    'SHOW_INDIVIDUAL_CHARTS': False
}

class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
                if req_symbol == "SOLUSDT": req_symbol = "SOLUSD"
                if req_symbol == "ADAUSDT": req_symbol = "ADAUSD"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        return df

class TrendBreakoutVolumeStrategy:
    def __init__(self):
        self.ema_period = 30
        self.breakout_period = 7
        self.vol_period = 6

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['EMA_30'] = ta.ema(df['Close'], length=self.ema_period)
        df['EMA_30_Rising'] = df['EMA_30'] > df['EMA_30'].shift(1)

        df['MAX_7'] = df['High'].rolling(self.breakout_period).max().shift(1)
        df['MIN_7'] = df['Low'].rolling(self.breakout_period).min().shift(1)

        df['ATR_7'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.breakout_period)
        df['NATR_7'] = (df['ATR_7'] / df['Close']) * 100
        df['NATR_7_Rising'] = df['NATR_7'] > df['NATR_7'].shift(1)

        df['TR'] = ta.true_range(df['High'], df['Low'], df['Close'])
        df['TR_MA_6'] = df['TR'].rolling(self.vol_period).mean()
        df['Vol_MA_6'] = df['Volume'].rolling(self.vol_period).mean()

        df['Effort'] = df['Volume'] / df['Vol_MA_6'].replace(0, np.nan)
        df['Result'] = df['TR'] / df['TR_MA_6'].replace(0, np.nan)
        df['Volume_Anomaly'] = df['Effort'] > df['Result']

        df.dropna(inplace=True)
        return df

class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.config = config
        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades = []
        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O, H, L, C = self.df['Open'].values, self.df['High'].values, self.df['Low'].values, self.df['Close'].values
        times = self.df.index
        ema_rising = self.df['EMA_30_Rising'].values
        max7, min7 = self.df['MAX_7'].values, self.df['MIN_7'].values
        natr_rising = self.df['NATR_7_Rising'].values
        vol_anomaly = self.df['Volume_Anomaly'].values

        fee, slip, pos_size_pct = self.config['FEE_RATE'], self.config['SLIPPAGE'], self.config['POSITION_SIZE_PCT']
        self.equity_curve[0] = self.capital; self.equity_curve[1] = self.capital

        for i in range(2, len(self.df)):
            if self.position == 1:
                current_stop = min7[i]
                if L[i] <= current_stop:
                    real_exit = min(O[i], current_stop) * (1 - slip)
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt
                    self.trades.append({'Date': times[i], 'Type': 'Long', 'Reason': 'TS Hit', 'Entry': self.entry_price, 'Exit': real_exit, 'PnL_Amt': pnl_amt})
                    self.position = 0
                    self.invested_amount = 0.0

            if self.position == 0:
                if ema_rising[i-1] and C[i-1] > max7[i-1] and natr_rising[i-1] and vol_anomaly[i-1]:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct
                    current_stop = min7[i]
                    if L[i] <= current_stop:
                        real_exit = current_stop * (1 - slip)
                        pnl_pct = (real_exit - self.entry_price) / self.entry_price
                        pnl_amt = self.invested_amount * (pnl_pct - fee * 2)
                        self.capital += pnl_amt
                        self.trades.append({'Date': times[i], 'Type': 'Long', 'Reason': 'Instant SL', 'Entry': self.entry_price, 'Exit': real_exit, 'PnL_Amt': pnl_amt})
                        self.position = 0
                        self.invested_amount = 0.0

            current_equity = self.capital
            if self.position == 1:
                current_equity += self.invested_amount * (C[i] / self.entry_price) - self.invested_amount
            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

class Evaluator:
    @staticmethod
    def process_results(equity, trades_df, initial_capital, symbol, interval, show_chart=False):
        if trades_df.empty:
            return None, None

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        annual_factor = 365 if interval in ['1d', 'D'] else (365*24 if interval == '1h' else (365*6 if interval == '4h' else 252))
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        stats = {
            'Symbol': symbol,
            'Timeframe': interval,
            'Trades': len(trades_df),
            'Winrate': f"{winrate*100:.2f}%",
            'Profit_Factor': round(profit_factor, 2),
            'Max_DD': f"{max_dd*100:.2f}%",
            'Net_PnL_$': round(net_pnl, 2),
            'Return': f"{total_return*100:.2f}%",
            'Sharpe': round(sharpe_ratio, 2)
        }

        if show_chart:
            fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])
            fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines", name="Equity", line=dict(color="#2962ff", width=1.5)), row=1, col=1)
            fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="Drawdown", line=dict(color="#d32f2f", width=1), fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'), row=2, col=1)
            fig.update_layout(title=f"Performance: {symbol} ({interval}) - Return: {total_return*100:.2f}%", template="plotly_white", height=500, margin=dict(l=40, r=40, t=40, b=40), showlegend=False)
            fig.show()

        return stats, equity

    @staticmethod
    def show_summary_dashboard(results_list, all_equities):
        if not results_list:
            print("No valid results to display.")
            return

        df_res = pd.DataFrame(results_list)

        df_res = df_res[df_res['Sharpe'] >= 1.0]
        df_res = df_res.sort_values(by='Sharpe', ascending=False).reset_index(drop=True)

        if df_res.empty:
            print("No strategies achieved a Sharpe Ratio >= 1.0")
            return

        print("\n" + "="*85)
        print(" PERFORMANCE RANKING (SHARPE RATIO >= 1.0) ")
        print("="*85)
        print(df_res.to_string(index=False))
        print("="*85 + "\n")

        fig = go.Figure()

        for index, row in df_res.iterrows():
            key = f"{row['Symbol']}_{row['Timeframe']}"
            if key in all_equities:
                eq_curve = all_equities[key]
                fig.add_trace(go.Scatter(
                    x=eq_curve.index,
                    y=eq_curve.values,
                    mode='lines',
                    name=f"{row['Symbol']} ({row['Timeframe']}) | SR: {row['Sharpe']}"
                ))

        fig.update_layout(
            title="Equity Curves Comparison (Top Performers)",
            xaxis_title="Date",
            yaxis_title="Account Balance ($)",
            template="plotly_white",
            hovermode="x unified",
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="right",
                x=1
            ),
            margin=dict(l=40, r=40, t=60, b=40),
            height=600
        )

        fig.update_yaxes(tickprefix="$", showgrid=True, gridcolor='#e6e6e6')
        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6')

        fig.show()

if __name__ == "__main__":
    all_performance_results = []
    all_equity_curves = {}

    print("Starting Multi-Timeframe Backtest...\n")

    for symbol in CONFIG['SYMBOLS']:
        for interval in CONFIG['INTERVALS']:
            print(f"Processing: {symbol} | Timeframe: {interval}")

            fetcher = BinanceDataFetcher(symbol, interval, CONFIG['START_DATE'], CONFIG['END_DATE'])
            df_raw = fetcher.fetch()

            if not df_raw.empty:
                my_strategy = TrendBreakoutVolumeStrategy()
                engine = BacktestEngine(df_raw, my_strategy, CONFIG)
                equity, trades = engine.run()

                stats, eq_series = Evaluator.process_results(
                    equity,
                    trades,
                    CONFIG['INITIAL_CAPITAL'],
                    symbol,
                    interval,
                    show_chart=CONFIG['SHOW_INDIVIDUAL_CHARTS']
                )

                if stats:
                    all_performance_results.append(stats)
                    all_equity_curves[f"{symbol}_{interval}"] = eq_series

            print("-" * 40)

    print("\nBacktest completed. Generating results...\n")
    Evaluator.show_summary_dashboard(all_performance_results, all_equity_curves)

Starting Multi-Timeframe Backtest...

Processing: BTCUSDT | Timeframe: 1h
----------------------------------------
Processing: BTCUSDT | Timeframe: 4h
----------------------------------------
Processing: BTCUSDT | Timeframe: 1d
----------------------------------------
Processing: ETHUSDT | Timeframe: 1h
----------------------------------------
Processing: ETHUSDT | Timeframe: 4h
----------------------------------------
Processing: ETHUSDT | Timeframe: 1d
----------------------------------------
Processing: SOLUSDT | Timeframe: 1h
----------------------------------------
Processing: SOLUSDT | Timeframe: 4h
----------------------------------------
Processing: SOLUSDT | Timeframe: 1d
----------------------------------------
Processing: BNBUSDT | Timeframe: 1h
----------------------------------------
Processing: BNBUSDT | Timeframe: 4h
----------------------------------------
Processing: BNBUSDT | Timeframe: 1d
----------------------------------------
Processing: ADAUSDT | Timeframe: 1h
--